In [0]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

kst = ZoneInfo("Asia/Seoul")

In [0]:
# =========================================================
# 1. Silver 기준 마지막 처리 시각
# =========================================================
SLV_TABLE = "hive_metastore.demo_airstatus_silver.SLV_fact_air_quality"

row = (
    spark.table(SLV_TABLE)
    .selectExpr("MAX(dataTime) AS last_dataTime")
    .collect()
)

last_dataTime = row[0]["last_dataTime"]

if last_dataTime is None:
    raise Exception("❌ Silver is empty. BACKFILL requires initial data.")

last_dataTime_kst = last_dataTime.replace(tzinfo=kst)
now_kst = datetime.now(kst)

print(f"[INFO] last_dataTime (KST): {last_dataTime_kst}")
print(f"[INFO] now (KST)          : {now_kst}")

In [0]:
# =========================================================
# 2. 관측소 목록
# =========================================================
stations = (
    spark.table("hive_metastore.demo_airstatus_bronze.BRZ_seoul_stations")
    .select("stationName")
    .rdd
    .map(lambda r: r["stationName"])
    .collect()
)

print(f"[INFO] Number of stations: {len(stations)}")

In [0]:
# =========================================================
# 3. API 설정
# =========================================================
SERVICE_KEY = "qz6MbARyTC9CL2GrNus/xLfgJolMh3LYaY+kT98k6wDcHDbTZ/vRAvw8JGmU9PnK25a7lM/ePpU1Dl7AsKAyXw=="
air_url = "http://apis.data.go.kr/B552584/ArpltnInforInqireSvc/getMsrstnAcctoRltmMesureDnsty"

all_rows = []
failed_stations = []

In [0]:
# =========================================================
# 4. 관측소별 수집 (ALL-OR-NOTHING)
# =========================================================
for station in stations:
    print(f"[INFO] Collecting station: {station}")
    station_rows = []

    params = {
        "serviceKey": SERVICE_KEY,
        "returnType": "json",
        "numOfRows": "1000",
        "pageNo": "1",
        "stationName": station,
        "dataTerm": "MONTH",
        "ver": "1.0"
    }

    try:
        r = requests.get(air_url, params=params, timeout=20)
        if r.status_code != 200:
            raise Exception(f"HTTP {r.status_code}")

        items = r.json()["response"]["body"]["items"]

        for i in items:
            raw_time = i.get("dataTime")
            if raw_time is None:
                continue

            # 24:00 처리
            if raw_time.endswith(" 24:00"):
                base_date = datetime.strptime(raw_time[:10], "%Y-%m-%d")
                dt_kst = base_date + timedelta(days=1)
            else:
                dt_kst = datetime.strptime(raw_time, "%Y-%m-%d %H:%M")

            dt_kst = dt_kst.replace(tzinfo=kst)

            # 증분 필터
            if last_dataTime_kst < dt_kst <= now_kst:
                station_rows.append({
                    "stationName": station,
                    "dataTime": raw_time,
                    "khaiValue": i.get("khaiValue"),
                    "khaiGrade": i.get("khaiGrade"),
                    "pm10Value": i.get("pm10Value"),
                    "pm25Value": i.get("pm25Value")
                })

        # ✅ 해당 관측소 성공
        all_rows.extend(station_rows)

    except Exception as e:
        print(f"❌ Station failed: {station}, error={e}")
        failed_stations.append(station)

In [0]:
# =========================================================
# 5. 실패 관측소 체크
# =========================================================
if failed_stations:
    print("❌ BACKFILL aborted due to station failures")
    print(f"❌ Failed stations: {failed_stations}")
    raise Exception("BACKFILL failed. No data written.")

print(f"[INFO] BACKFILL rows collected: {len(all_rows)}")

if len(all_rows) == 0:
    print("⚠️ No new data to backfill.")
    dbutils.notebook.exit("NO_DATA")

In [0]:
# =========================================================
# 6. TEMP overwrite
# =========================================================
air_pdf = pd.DataFrame(all_rows)
air_sdf = spark.createDataFrame(air_pdf)

air_sdf.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(
        "hive_metastore.demo_airstatus_bronze.BRZ_temp_seoul_air_quality_month"
    )

In [0]:
# =========================================================
# 7. MAIN append (성공 시 단 1회)
# =========================================================
spark.sql("""
    INSERT INTO hive_metastore.demo_airstatus_bronze.BRZ_seoul_air_quality_hourly
    SELECT * FROM hive_metastore.demo_airstatus_bronze.BRZ_temp_seoul_air_quality_month
""")

print("✅ BACKFILL completed successfully.")